# CatAI GPU training (Kaggle)

Runs the existing CatAI trainer on a Kaggle GPU without changing model dimensions, context length, datasets, sampling ratios, token budgets, optimizer hyperparameters, or epoch counts. Attach a Kaggle Dataset named `catai-training-data` containing the files listed in `docs/KAGGLE_GPU_TRAINING.md`.

In [ ]:
from pathlib import Path
import os, subprocess, sys, time
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is attached. In notebook Settings → Session options, set Accelerator to GPU.")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("PyTorch:", torch.__version__)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


In [ ]:
# Kaggle Secrets → Add a secret named GITHUB_TOKEN with read-only access to bananakitssu/catai.
import base64
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
repo_dir = Path("/kaggle/working/catai")
if not repo_dir.exists():
    git_env = os.environ.copy()
    auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
    git_env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}",
    })
    result = subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", "https://github.com/bananakitssu/catai.git", str(repo_dir)], env=git_env, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("Could not clone CatAI. Check that GITHUB_TOKEN has read-only access to the private repository.")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(repo_dir)], check=True)
os.chdir(repo_dir)
print("CatAI source:", repo_dir)


In [ ]:
# Attach the Kaggle Dataset `catai-training-data` to this notebook.
DATA = Path("/kaggle/input/catai-training-data")
def data_file(name):
    for candidate in (DATA / name, DATA / "data" / name):
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Missing {name} in the attached catai-training-data dataset")
required = [
    "catai.pt", "oasst_chat.jsonl", "instruction_chat.jsonl",
    "smol_smoltalk_chat.jsonl", "blended_skill_talk_chat.jsonl",
    "ultrachat_200k_chat.jsonl", "expanded_sft_chat.jsonl",
    "markdown_sft_chat.jsonl",
]
files = {name: data_file(name) for name in required}
print("All SFT inputs found.")
for name, path in files.items():
    print(f"{name}: {path.stat().st_size / (1024**2):.1f} MiB")


In [ ]:
# These are the same SFT settings used by the repository workflow.
# Only the execution device changes from CPU to CUDA.
command = [
    sys.executable, "-m", "catai.sft_cli",
    "--base-checkpoint", str(files["catai.pt"]),
    "--dataset", str(files["oasst_chat.jsonl"]),
    "--personality-dataset", str(files["instruction_chat.jsonl"]),
    "--personality-ratio", "0.03",
    "--smol-smoltalk-dataset", str(files["smol_smoltalk_chat.jsonl"]),
    "--smol-smoltalk-ratio", "0.10",
    "--blended-skill-talk-dataset", str(files["blended_skill_talk_chat.jsonl"]),
    "--blended-skill-talk-ratio", "0.05",
    "--ultrachat-dataset", str(files["ultrachat_200k_chat.jsonl"]),
    "--ultrachat-ratio", "0.25",
    "--expanded-chat-dataset", str(files["expanded_sft_chat.jsonl"]),
    "--expanded-chat-ratio", "0.15",
    "--markdown-dataset", str(files["markdown_sft_chat.jsonl"]),
    "--markdown-ratio", "0.05",
    "--samples-per-epoch", "6000",
    "--max-validation-examples", "500",
    "--checkpoint", "/kaggle/working/catai-sft.pt",
    "--epochs", "3",
    "--batch-size", "2",
    "--learning-rate", "2e-5",
    "--minimum-tokens-per-parameter", "20",
    "--grad-clip", "1.0",
    "--validation-split", "0.05",
    "--patience", "2",
    "--sequence-length", "1024",
    "--seed", "42",
    "--device", "cuda",
    "--precision", "auto",
    "--recovery-checkpoint", "/kaggle/working/catai-sft-recovery.pt",
    "--checkpoint-every-batches", "50",
]
resume = next((candidate for candidate in (DATA / "catai-sft-recovery.pt", DATA / "data" / "catai-sft-recovery.pt") if candidate.is_file()), None)
if resume is not None:
    command.extend(["--resume-from", str(resume)])
print("Starting unchanged SFT workload on", torch.cuda.get_device_name(0))
started = time.perf_counter()
subprocess.run(command, cwd=repo_dir, check=True)
torch.cuda.synchronize()
print(f"Training elapsed: {(time.perf_counter() - started) / 3600:.2f} hours")
subprocess.run([sys.executable, "scripts/export_inference.py", "--checkpoint", "/kaggle/working/catai-sft.pt", "--output", "/kaggle/working/catai-sft.npz"], cwd=repo_dir, check=True)
print("Outputs: catai-sft.pt, catai-sft-recovery.pt, catai-sft.npz")
